# Analisis Komparatif Algoritma XGBoost dan CatBoost
## untuk Sistem Deteksi Intrusi pada Dataset NF-UNSW-NB15-v3

Notebook ini dirancang untuk membandingkan dua model gradient boosting (XGBoost, CatBoost) pada skenario IDS secara **fair** (split data dan preprocessing yang sama).


## 1) Tujuan Komparasi
- Membandingkan performa **XGBoost** dan **CatBoost** pada dataset **NF-UNSW-NB15-v3**.
- Menentukan model terbaik untuk IDS berdasarkan metrik utama.

### Metrik utama
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Waktu latih (training time)
- Waktu inferensi (inference time)


In [1]:
# 2) Setup Eksperimen (Seed, Environment, Versi Library)
import os
import time
import random
import warnings
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

SEED = 42
TEST_SIZE = 0.2
RANDOM_STATE = SEED

random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
RUN_ENV = "Kaggle Notebook" if IS_KAGGLE else "Local/Other"


def detect_xgboost_gpu():
    try:
        x_dummy = np.random.rand(128, 8)
        y_dummy = np.random.randint(0, 3, 128)
        model = XGBClassifier(
            n_estimators=1,
            max_depth=2,
            objective="multi:softprob",
            num_class=3,
            eval_metric="mlogloss",
            tree_method="hist",
            device="cuda",
            random_state=SEED,
        )
        model.fit(x_dummy, y_dummy)
        return True, ""
    except Exception as e:
        return False, str(e)


def detect_catboost_gpu():
    try:
        x_dummy = np.random.rand(128, 8)
        y_dummy = np.random.randint(0, 3, 128)
        model = CatBoostClassifier(
            iterations=1,
            depth=2,
            learning_rate=0.1,
            loss_function="MultiClass",
            task_type="GPU",
            devices="0",
            random_seed=SEED,
            verbose=False,
        )
        model.fit(x_dummy, y_dummy)
        return True, ""
    except Exception as e:
        return False, str(e)


XGB_GPU_ENABLED, xgb_gpu_err = detect_xgboost_gpu()
CAT_GPU_ENABLED, cat_gpu_err = detect_catboost_gpu()

GPU_PARAMS = {
    "XGBoost": {"tree_method": "hist", "device": "cuda"} if XGB_GPU_ENABLED else {"tree_method": "hist", "device": "cpu"},
    "CatBoost": {"task_type": "GPU", "devices": "0"} if CAT_GPU_ENABLED else {"task_type": "CPU"},
}

print("✅ Setup selesai")
print(f"Python      : {platform.python_version()}")
print(f"OS          : {platform.system()} {platform.release()}")
print(f"Environment : {RUN_ENV}")
print(f"Seed        : {SEED}")
print(f"XGBoost GPU : {'AKTIF' if XGB_GPU_ENABLED else 'CPU fallback'}")
if not XGB_GPU_ENABLED and xgb_gpu_err:
    print(f"  └─ Alasan fallback XGBoost: {xgb_gpu_err}")
print(f"CatBoost GPU: {'AKTIF' if CAT_GPU_ENABLED else 'CPU fallback'}")
if not CAT_GPU_ENABLED and cat_gpu_err:
    print(f"  └─ Alasan fallback CatBoost: {cat_gpu_err}")


✅ Setup selesai
Python      : 3.12.12
OS          : Linux 6.6.113+
Environment : Kaggle Notebook
Seed        : 42


In [2]:
# Cek versi library agar eksperimen reproducible
import sklearn
import xgboost
import catboost

print("Versi library:")
print(f"- pandas    : {pd.__version__}")
print(f"- numpy     : {np.__version__}")
print(f"- scikit-learn: {sklearn.__version__}")
print(f"- xgboost   : {xgboost.__version__}")
print(f"- catboost  : {catboost.__version__}")


Versi library:
- pandas    : 2.3.3
- numpy     : 2.0.2
- scikit-learn: 1.6.1
- xgboost   : 3.2.0
- catboost  : 1.2.10


## 3) Load Data Singkat (Minimum)
Notebook akan mencoba menemukan file dataset otomatis dari beberapa lokasi umum.


In [3]:
# 3.1 Load dataset NF-UNSW-NB15-v3 (Kaggle-friendly)
candidate_paths = []

if IS_KAGGLE:
    print("🔍 Mencari dataset di /kaggle/input ...")
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            name = filename.lower()
            if name.endswith(".csv") and "nf-unsw-nb15-v3" in name and "features" not in name:
                candidate_paths.append(Path(dirname) / filename)

# Fallback lokal (tetap mendukung non-Kaggle)
candidate_paths.extend([
    Path("NF-UNSW-NB15-v3.csv"),
    Path("./data/NF-UNSW-NB15-v3.csv"),
    Path("./dataset/NF-UNSW-NB15-v3.csv"),
])

# Tambahan: cari file csv dengan nama yang mengandung NF-UNSW-NB15-v3
for p in Path(".").glob("**/*NF-UNSW-NB15-v3*.csv"):
    candidate_paths.append(p)

# Deduplicate sambil menjaga urutan
unique_candidates = []
seen = set()
for p in candidate_paths:
    p_str = str(p)
    if p_str not in seen:
        unique_candidates.append(p)
        seen.add(p_str)

dataset_path = next((p for p in unique_candidates if p.exists() and p.is_file()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Dataset NF-UNSW-NB15-v3 tidak ditemukan. Tambahkan dataset ke Kaggle Notebook (Input) "
        "atau letakkan file CSV di root project/folder data/dataset."
    )

df = pd.read_csv(dataset_path)
print(f"✅ Dataset ditemukan: {dataset_path}")
print(f"Shape data: {df.shape}")

df.head(3)


🔍 Mencari dataset di /kaggle/input ...
✅ Dataset ditemukan: /kaggle/input/datasets/rachmanantaibnufajar/nf-unsw-nb15-v3/NF-UNSW-NB15-v3.csv
Shape data: (2365424, 55)


,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign


In [8]:
# 3.2 Cek ukuran data & distribusi label (imbalanced check)
print("Info dataframe:")
print(df.info())

# Deteksi kolom label secara fleksibel
label_candidates = ["Attack"]
label_col = next((c for c in label_candidates if c in df.columns), None)

if label_col is None:
    raise ValueError(f"Kolom label tidak ditemukan. Kandidat yang dicek: {label_candidates}")

print(f"\n✅ Kolom label yang digunakan: {label_col}")
print("\nDistribusi label:")
display(df[label_col].value_counts(dropna=False).to_frame("count"))


Info dataframe:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2350609 entries, 0 to 2350608
Data columns (total 55 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   FLOW_START_MILLISECONDS      int64  
 1   FLOW_END_MILLISECONDS        int64  
 2   IPV4_SRC_ADDR                object 
 3   L4_SRC_PORT                  int64  
 4   IPV4_DST_ADDR                object 
 5   L4_DST_PORT                  int64  
 6   PROTOCOL                     int64  
 7   L7_PROTO                     float64
 8   IN_BYTES                     int64  
 9   IN_PKTS                      int64  
 10  OUT_BYTES                    int64  
 11  OUT_PKTS                     int64  
 12  TCP_FLAGS                    int64  
 13  CLIENT_TCP_FLAGS             int64  
 14  SERVER_TCP_FLAGS             int64  
 15  FLOW_DURATION_MILLISECONDS   int64  
 16  DURATION_IN                  int64  
 17  DURATION_OUT                 int64  
 18  MIN_TTL                   

,count
Attack,
Benign,2222930
Exploits,42744
Fuzzers,33816
Generic,19651
Reconnaissance,17074
DoS,5971
Backdoor,4658
Shellcode,2381
Analysis,1226


## 4) Definisi Skenario & Split Data
- Skenario akan otomatis terdeteksi:
  - **Binary** jika jumlah kelas = 2
  - **Multiclass** jika jumlah kelas > 2
- Semua model menggunakan split train/test yang sama (prinsip fairness).


In [9]:
# 4.1 Pembersihan data minimum + split
# Hapus duplikasi baris (minimum cleaning)
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=[label_col]).copy()
y_raw = df[label_col].copy()

# Drop kolom ID bila ada (umum pada dataset network flow)
id_like_cols = [c for c in X.columns if c.lower() in {"id", "flow_id", "index"}]
if id_like_cols:
    X = X.drop(columns=id_like_cols)
    print(f"Kolom ID dihapus: {id_like_cols}")

# Encoding label
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(y_raw.astype(str))
class_names = list(y_encoder.classes_)

n_classes = len(np.unique(y))
if n_classes < 3:
    raise ValueError(f"Notebook ini difokuskan untuk multiclass, tetapi hanya terdeteksi {n_classes} kelas.")
scenario = "multiclass"
print(f"Skenario eksperimen: {scenario} ({n_classes} kelas)")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


Skenario eksperimen: multiclass (10 kelas)
Train shape: (1880487, 54), Test shape: (470122, 54)


## 5) Preprocessing Fitur Kategorikal Native untuk Semua Model
- Imputasi missing value numerik (median)
- Imputasi missing value kategorikal (token `MISSING`)
- Menjaga dtype `category` agar bisa dimanfaatkan native oleh model

> Catatan fairness: seluruh model dilatih pada split data yang sama dengan skema imputasi yang sama.


In [10]:
# 5.1 Preprocessing tanpa one-hot (native categorical) + class weighting multiclass
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

# Imputasi numerik berbasis train
if numeric_cols:
    num_imputer = SimpleImputer(strategy="median")
    X_train_prep[numeric_cols] = num_imputer.fit_transform(X_train_prep[numeric_cols])
    X_test_prep[numeric_cols] = num_imputer.transform(X_test_prep[numeric_cols])

# Imputasi + casting kategorikal ke dtype category agar native categorical tetap aktif
for col in categorical_cols:
    train_col = X_train_prep[col].astype("string").fillna("MISSING")
    test_col = X_test_prep[col].astype("string").fillna("MISSING")

    train_categories = pd.Index(train_col.unique())
    if "UNKNOWN" not in train_categories:
        train_categories = train_categories.append(pd.Index(["UNKNOWN"]))

    unseen_mask = ~test_col.isin(train_categories)
    test_col = test_col.mask(unseen_mask, "UNKNOWN")

    X_train_prep[col] = pd.Categorical(train_col, categories=train_categories)
    X_test_prep[col] = pd.Categorical(test_col, categories=train_categories)

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_map = {int(cls): float(w) for cls, w in zip(classes, class_weights)}
sample_weight_train = np.array([class_weight_map[y] for y in y_train], dtype=float)

print("✅ Preprocessing selesai (native categorical)")
print(f"Jumlah fitur awal        : {X_train.shape[1]}")
print(f"Jumlah fitur kategorikal : {len(categorical_cols)}")
print(f"Jumlah fitur numerik     : {len(numeric_cols)}")
print(f"Contoh class_weight      : {dict(list(class_weight_map.items())[:5])}")


ValueError: Input X contains infinity or a value too large for dtype('float64').

## 6) Definisi Baseline Dua Model (Multiclass)
- XGBoost dengan `enable_categorical=True`
- CatBoost menggunakan `cat_features` saat fit


In [ ]:
# 6.1 Baseline model multiclass
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    num_class=n_classes,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    enable_categorical=True,
    **GPU_PARAMS["XGBoost"],
)

cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function="MultiClass",
    random_seed=SEED,
    verbose=0,
    **GPU_PARAMS["CatBoost"],
)

models = {
    "XGBoost": xgb_model,
    "CatBoost": cat_model,
}

print("✅ Baseline model siap:", list(models.keys()))


## 7) Pelatihan & Prediksi (dengan pencatatan waktu)


In [ ]:
# 7.1 Train + predict + timing
def normalize_pred_output(y_pred):
    """Normalisasi output prediksi agar konsisten 1D (lintas library)."""
    return np.asarray(y_pred).ravel()


results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    print(f"\n🚀 Training {name}...")

    fit_kwargs = {"sample_weight": sample_weight_train}
    if name == "CatBoost":
        fit_kwargs["cat_features"] = categorical_cols

    try:
        t0 = time.perf_counter()
        model.fit(X_train_prep, y_train, **fit_kwargs)
        train_time = time.perf_counter() - t0

        t1 = time.perf_counter()
        y_pred = normalize_pred_output(model.predict(X_test_prep))
        infer_time_total = time.perf_counter() - t1
        infer_time_per_sample_ms = (infer_time_total / len(y_test)) * 1000

        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test_prep)
        else:
            y_proba = None

        # Metrik klasifikasi multiclass
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

        if y_proba is None:
            roc_auc = np.nan
        else:
            try:
                roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")
            except ValueError:
                roc_auc = np.nan

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "ROC-AUC": roc_auc,
            "Train Time (s)": train_time,
            "Inference Total (s)": infer_time_total,
            "Inference / Sample (ms)": infer_time_per_sample_ms,
            "Error": np.nan,
        })

        trained_models[name] = model
        predictions[name] = y_pred
        probabilities[name] = y_proba

        print(f"✅ {name} selesai | F1={f1:.4f} | Recall={rec:.4f} | Train={train_time:.3f}s")

    except Exception as e:
        results.append({
            "Model": name,
            "Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan,
            "ROC-AUC": np.nan,
            "Train Time (s)": np.nan,
            "Inference Total (s)": np.nan,
            "Inference / Sample (ms)": np.nan,
            "Error": str(e),
        })
        print(f"❌ {name} gagal dilatih: {e}")

if not trained_models:
    raise RuntimeError("Semua model gagal dilatih. Periksa dataset, versi library, dan konfigurasi GPU/CPU.")


## 8) Evaluasi Kuantitatif + Cross-Validation


In [ ]:
# 8.1 Cross-validation multiclass (weighted F1)
cv_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name in trained_models.keys():
    base_model = models[name]
    fold_scores = []
    for tr_idx, val_idx in cv.split(X_train_prep, y_train):
        X_tr = X_train_prep.iloc[tr_idx]
        X_val = X_train_prep.iloc[val_idx]
        y_tr = y_train[tr_idx]
        y_val = y_train[val_idx]

        fold_classes = np.unique(y_tr)
        fold_weights = compute_class_weight(class_weight="balanced", classes=fold_classes, y=y_tr)
        fold_map = {int(cls): float(w) for cls, w in zip(fold_classes, fold_weights)}
        fold_sample_weight = np.array([fold_map[y] for y in y_tr], dtype=float)

        model_cv = clone(base_model)
        fit_kwargs = {"sample_weight": fold_sample_weight}
        if name == "CatBoost":
            fit_kwargs["cat_features"] = categorical_cols

        model_cv.fit(X_tr, y_tr, **fit_kwargs)
        y_val_pred = normalize_pred_output(model_cv.predict(X_val))
        fold_f1 = f1_score(y_val, y_val_pred, average="weighted", zero_division=0)
        fold_scores.append(fold_f1)

    cv_rows.append({
        "Model": name,
        "CV F1 (mean)": float(np.mean(fold_scores)),
        "CV F1 (std)": float(np.std(fold_scores)),
    })

cv_results_df = pd.DataFrame(cv_rows)

# Tabel komparasi utama + ranking
all_results_df = pd.DataFrame(results)
failed_results_df = all_results_df[all_results_df["Error"].notna()].copy()
results_df = all_results_df[all_results_df["Error"].isna()].copy()

if results_df.empty:
    raise RuntimeError("Tidak ada model yang berhasil dilatih, sehingga komparasi tidak dapat dilanjutkan.")

results_df = results_df.merge(cv_results_df, on="Model", how="left")
results_df = results_df.sort_values(by=["F1", "Recall"], ascending=False).reset_index(drop=True)
results_df["Rank (F1->Recall)"] = np.arange(1, len(results_df) + 1)

cols_order = [
    "Rank (F1->Recall)", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC",
    "CV F1 (mean)", "CV F1 (std)",
    "Train Time (s)", "Inference Total (s)", "Inference / Sample (ms)"
]
results_df = results_df[cols_order]

print("📊 Tabel Komparasi Utama")
display(results_df)

if not failed_results_df.empty:
    print("\n⚠️ Model yang gagal (tidak masuk ranking):")
    display(failed_results_df[["Model", "Error"]])

best_model_name = results_df.iloc[0]["Model"]
print(f"\n🏆 Model terbaik berdasarkan prioritas F1/Recall: {best_model_name}")


In [ ]:
# 8.2 Laporan klasifikasi per model
for name in trained_models.keys():
    print("\n" + "="*80)
    print(f"CLASSIFICATION REPORT - {name}")
    print("="*80)
    print(classification_report(y_test, predictions[name], target_names=[str(c) for c in class_names], zero_division=0))


## 9) Visualisasi Perbandingan
- Bar chart metrik antar model
- Confusion matrix per model
- Grafik waktu komputasi


In [ ]:
# 9.1 Bar chart metrik utama antar model
plot_df = results_df.copy()
plot_df = plot_df.set_index("Model")

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
ax = plot_df[metrics_to_plot].plot(kind="bar", figsize=(12, 6))
ax.set_title("Perbandingan Metrik Utama Antar Model")
ax.set_ylabel("Skor")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# 9.2 Confusion matrix per model
model_names = list(trained_models.keys())
n_models = len(model_names)
fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, model_names):
    cm = confusion_matrix(y_test, predictions[name])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=[str(c) for c in class_names],
        yticklabels=[str(c) for c in class_names],
    )
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")

plt.tight_layout()
plt.show()


In [ ]:
# 9.3 Grafik waktu komputasi
time_cols = ["Train Time (s)", "Inference / Sample (ms)"]
ax = results_df.set_index("Model")[time_cols].plot(kind="bar", figsize=(10, 5), color=["#1f77b4", "#ff7f0e"])
ax.set_title("Perbandingan Waktu Komputasi")
ax.set_ylabel("Waktu")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10) Analisis Hasil Komparatif
- Model paling akurat
- Model paling cepat
- Trade-off performa vs efisiensi
- Kesesuaian untuk implementasi IDS real-time


In [ ]:
# 10.1 Ringkasan analisis otomatis
best_accuracy_model = results_df.sort_values("Accuracy", ascending=False).iloc[0]["Model"]
fastest_train_model = results_df.sort_values("Train Time (s)", ascending=True).iloc[0]["Model"]
fastest_infer_model = results_df.sort_values("Inference / Sample (ms)", ascending=True).iloc[0]["Model"]

print("📌 Ringkasan Analisis Komparatif")
print(f"- Model paling akurat              : {best_accuracy_model}")
print(f"- Model training tercepat         : {fastest_train_model}")
print(f"- Model inferensi tercepat        : {fastest_infer_model}")
print(f"- Model terbaik (prioritas F1/Recall): {best_model_name}")

print("\nTrade-off performa vs efisiensi:")
for _, row in results_df.iterrows():
    print(
        f"  • {row['Model']}: F1={row['F1']:.4f}, Recall={row['Recall']:.4f}, "
        f"CV F1={row['CV F1 (mean)']:.4f}±{row['CV F1 (std)']:.4f}, "
        f"Train={row['Train Time (s)']:.3f}s, Infer/sample={row['Inference / Sample (ms)']:.4f}ms"
    )


## 11) Kesimpulan Komparasi
Gunakan sel berikut untuk menghasilkan kesimpulan otomatis berbasis metrik, lalu sesuaikan narasi akhir sesuai konteks penelitian.


In [ ]:
# 11.1 Kesimpulan otomatis
winner = results_df.iloc[0]

print("="*80)
print("KESIMPULAN KOMPARASI")
print("="*80)
print(
    f"Model terbaik pada eksperimen ini adalah {winner['Model']} "
    f"(berdasarkan prioritas F1/Recall)."
)
print(
    f"Nilai utama: Accuracy={winner['Accuracy']:.4f}, Precision={winner['Precision']:.4f}, "
    f"Recall={winner['Recall']:.4f}, F1={winner['F1']:.4f}, ROC-AUC={winner['ROC-AUC']:.4f}."
)
print("\nJustifikasi:")
print("- Pemilihan model didasarkan pada ranking F1 lalu Recall untuk konteks IDS.")
print("- Waktu training dan inferensi turut dipertimbangkan untuk kebutuhan real-time.")

print("\nCatatan keterbatasan komparasi:")
print("- Baseline hyperparameter belum dilakukan tuning ekstensif.")
print("- Variansi antarfold perlu dipertimbangkan bersama metrik hold-out test.")
print("- Hasil sensitif terhadap kualitas data dan distribusi kelas pada NF-UNSW-NB15-v3.")
